In [1]:
#%pip install azure-ai-ml

In [2]:
# from azure.ai.ml.entities import IdentityConfiguration

# # 1. Fetch your existing auto-scaling cluster
# cluster = ml_client.compute.get("trading-cluster")

# # 2. Upgrade it to have its own System Assigned Identity
# cluster.identity = IdentityConfiguration(type="system_assigned")

# # 3. Apply the update to the cloud
# ml_client.compute.begin_create_or_update(cluster).result()
# print("✅ Compute Cluster successfully upgraded with an Identity!")

In [3]:
# # Fetch the cluster details
# cluster = ml_client.compute.get("trading-cluster")

# # Print the secret Object ID
# print(f"🎯 Copy this Object ID: {cluster.identity.principal_id}")

In [4]:
from azure.ai.ml import MLClient, command, dsl, Input, Output
from azure.ai.ml.entities import RecurrenceTrigger, RecurrencePattern, JobSchedule, Environment
from azure.identity import DefaultAzureCredential

# 1. Authenticate
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id="26b57ef9-1628-4837-a014-81f6424512c1",
    resource_group_name="rg-trading-bot-dev",
    workspace_name="ml-trading-workspace"
)

ml_env = Environment(
    name="trading-cluster-env",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml"
)

TARGET_COMPUTE = "trading-cluster"

# 3A. STEP 1: ETL
etl_component = command(
    code="./",  
    command="python etl_pipeline.py && echo 'done' > ${{outputs.signal_file}}",
    environment=ml_env,
    compute=TARGET_COMPUTE,
    display_name="Step 1: Data Engineering & Clustering",
    name="step_1_etl",
    outputs={"signal_file": Output(type="uri_file")}
    # Notice we removed the identity configuration entirely!
)

# 3B. STEP 2: AI Agent
ai_component = command(
    code="./",  
    command="python ai_agent.py", 
    environment=ml_env,
    compute=TARGET_COMPUTE,
    display_name="Step 2: Agentic Thesis Generation",
    name="step_2_ai_agent",
    inputs={"wait_for_signal": Input(type="uri_file")}, 
    environment_variables={
        "KEY_VAULT_URL": "https://kv-ml-trading-workspace.vault.azure.net/"
    }
    # Notice we removed the identity configuration entirely!
)

# 4. Construct the Dependency Graph
@dsl.pipeline(name="decoupled_quant_pipeline", description="Enterprise MLOps Pipeline")
def my_quant_pipeline():
    # Instantiate Step 1
    step_1 = etl_component()
    
    # --- THE FIX: EXPLICIT NODE NAMING ---
    step_1.name = "step_1_etl"
    step_1.display_name = "Step 1: Data Engineering & Clustering"
    
    # Instantiate Step 2 (passing the dummy signal)
    step_2 = ai_component(wait_for_signal=step_1.outputs.signal_file)
    
    # --- THE FIX: EXPLICIT NODE NAMING ---
    step_2.name = "step_2_ai_agent"
    step_2.display_name = "Step 2: Agentic Thesis Generation"

pipeline_job = my_quant_pipeline()
pipeline_job.settings.default_compute = TARGET_COMPUTE

job_schedule = JobSchedule(
    name="daily_market_clustering_schedule",
    trigger=RecurrenceTrigger(frequency="day", interval=1, schedule=RecurrencePattern(hours=21, minutes=30)),
    create_job=pipeline_job
)

ml_client.schedules.begin_create_or_update(job_schedule)
print(f"✅ Master Pipeline Deployed! Compute Cluster Identity will handle auth natively.")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

✅ Master Pipeline Deployed! Compute Cluster Identity will handle auth natively.
..